# Adversarial robustness metric + defence-sweep demo

This notebook uses **synthetic score arrays** to demonstrate the reconstructed evaluation contract. It does not reproduce or claim the historical ImageNet numbers.

In [ ]:
import numpy as np
from adversarial_portfolio.metrics import classification_summary, defence_summary

clean_scores = np.array([[.90,.10],[.15,.85],[.80,.20],[.30,.70]])
adversarial_scores = np.array([[.20,.80],[.25,.75],[.40,.60],[.35,.65]])
labels = np.array([0,1,0,1])
classification_summary(clean_scores, adversarial_scores, labels)


## Evaluate a defence without hiding clean-accuracy cost

A defence is useful only if robustness gain is considered together with its effect on clean inputs.

In [ ]:
def_clean_scores = np.array([[.82,.18],[.55,.45],[.70,.30],[.40,.60]])
def_adv_scores = np.array([[.65,.35],[.40,.60],[.58,.42],[.45,.55]])
defence_summary(clean_scores, adversarial_scores, def_clean_scores, def_adv_scores, labels)


## Parameter-sweep contract

For real image experiments, the `predictor` callable owns the complete model preprocessing pipeline. Clean and adversarial inputs are transformed and predicted through that same contract, preventing the preprocessing mismatch found in the historical notebook.

In [ ]:
from adversarial_portfolio.sweeps import run_defence_sweep

inputs = np.array([[2.,0.],[0.,2.]])
adversarial = np.array([[0.,2.],[0.,2.]])

def predictor(x):
    z = np.asarray(x, dtype=float)
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

run_defence_sweep(
    parameters=[1.0, 0.75, 0.5],
    clean_inputs=inputs,
    adversarial_inputs=adversarial,
    true_labels=[0,1],
    predictor=predictor,
    transform_factory=lambda scale: (lambda x: np.asarray(x) * scale),
    parameter_name='scale',
)


## Interpretation boundary

A fixed-attack sweep is diagnostic, not a robustness certificate. A final defence should be evaluated against an adaptive attacker under a predeclared threat model.